# 10 · 权重初始化 + 训练 MLP 识别 MNIST

> **本节属于 Part 4 · 神经网络抽象 nn。这是 minitorch 的第一个"真任务"！**

万事俱备。本节我们先弄清一个常被忽视却至关重要的细节——**权重初始化**，然后用 minitorch **真刀真枪地训练一个 MLP 识别 MNIST 手写数字**，目标准确率 95%+，并与 PyTorch 对照。

## 学习目标

- 理解**权重初始化为什么重要**（梯度消失/爆炸），认识 Xavier / He 初始化
- 用 `minitorch.nn` 搭建并训练一个 MLP 分类 MNIST（目标 **95%+**）
- 掌握标准的 **mini-batch 训练循环**
- 与等价的 PyTorch 实现对照

## 1. 为什么初始化很重要

如果初始权重太小，信号逐层衰减 → 激活和梯度**消失**；太大则**爆炸**或让激活饱和。好的初始化让每一层的激活方差保持稳定。

做个实验：让随机信号穿过 8 层 `tanh` 网络，看不同初始化下各层激活的标准差。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad
from minitorch.nn import init

def layer_stds(init_fn, depth=8, width=256):
    x = np.random.randn(256, width)
    stds = []
    for _ in range(depth):
        W = init_fn((width, width))
        x = np.tanh(x @ W)
        stds.append(x.std())
    return stds

np.random.seed(0)
small = layer_stds(lambda s: np.random.randn(*s) * 0.01)   # 太小
xavier = layer_stds(init.xavier_uniform)                    # Xavier
for name, stds in [("×0.01 (太小)", small), ("Xavier", xavier)]:
    print(f"{name:14s} 各层激活 std:", [f"{v:.3f}" for v in stds])

可以看到：`×0.01` 初始化下激活迅速衰减到接近 0（信号消失）；而 **Xavier** 让各层 std 保持稳定。这就是 `minitorch.nn.Linear` 默认采用合理初始化（Kaiming/He）的原因。

## 2. 加载 MNIST

`load_mnist` 会自动下载并缓存到 `data/`（只下一次）。为了在 CPU 上分钟级跑完，我们取 2 万训练样本。

In [ ]:
(X_tr, y_tr), (X_te, y_te) = minitorch.utils.load_mnist(n_train=20000, n_test=5000)
print("训练集:", X_tr.shape, " 测试集:", X_te.shape)

fig, axes = plt.subplots(1, 8, figsize=(11, 1.6))
for i, ax in enumerate(axes):
    ax.imshow(X_tr[i].reshape(28, 28), cmap="gray")
    ax.set_title(str(y_tr[i])); ax.axis("off")
plt.suptitle("MNIST samples"); plt.tight_layout(); plt.show()

## 3. 搭建并训练

模型：`784 → 128 → ReLU → 10`。训练用标准的 mini-batch SGD 循环——和 nb01 的核心循环一模一样，只是现在用 `model` / `loss_fn` 优雅地表达。

In [ ]:
import time

minitorch.set_seed(0)
model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
loss_fn = nn.CrossEntropyLoss()
lr, bs, epochs = 0.2, 64, 15

def accuracy(X, y):
    with no_grad():
        pred = model(Tensor(X)).data.argmax(axis=1)
    return (pred == y).mean()

idx = np.arange(len(X_tr))
hist = {"loss": [], "acc": []}
t0 = time.time()
for ep in range(epochs):
    np.random.shuffle(idx)
    ep_loss = 0.0
    for i in range(0, len(X_tr), bs):
        b = idx[i:i + bs]
        model.zero_grad()                              # 梯度清零
        loss = loss_fn(model(Tensor(X_tr[b])), y_tr[b])  # 前向 + 损失
        loss.backward()                                # 反向
        for p in model.parameters():                   # SGD 更新
            p.data -= lr * p.grad
        ep_loss += float(loss.data)
    hist["loss"].append(ep_loss / (len(X_tr) // bs))
    hist["acc"].append(accuracy(X_te, y_te))
    if ep % 3 == 0 or ep == epochs - 1:
        print(f"epoch {ep:2d}  train_loss {hist['loss'][-1]:.3f}  test_acc {hist['acc'][-1]*100:.2f}%")

print(f"\n最终测试准确率: {hist['acc'][-1]*100:.2f}%   (用时 {time.time()-t0:.1f}s)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(hist["loss"]); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=0.3)
ax[1].plot([a*100 for a in hist["acc"]]); ax[1].set_title("Test accuracy (%)"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 看看预测效果

In [ ]:
with no_grad():
    pred = model(Tensor(X_te[:10])).data.argmax(axis=1)
fig, axes = plt.subplots(1, 10, figsize=(13, 1.6))
for i, ax in enumerate(axes):
    ax.imshow(X_te[i].reshape(28, 28), cmap="gray")
    color = "green" if pred[i] == y_te[i] else "red"
    ax.set_title(f"{pred[i]}", color=color); ax.axis("off")
plt.suptitle("Predictions (green=correct, red=wrong)"); plt.tight_layout(); plt.show()

## 4. PyTorch 对照

同样的模型与任务，用 PyTorch 训练几轮，准确率应当相当（实现细节不同，比的是量级与趋势）。

In [ ]:
import torch
import torch.nn as tnn

torch.manual_seed(0)
tmodel = tnn.Sequential(tnn.Linear(784, 128), tnn.ReLU(), tnn.Linear(128, 10))
topt = torch.optim.SGD(tmodel.parameters(), lr=0.2)
tloss = tnn.CrossEntropyLoss()
Xtr_t = torch.tensor(X_tr, dtype=torch.float32); ytr_t = torch.tensor(y_tr)

for ep in range(5):
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(Xtr_t), 64):
        b = perm[i:i + 64]
        topt.zero_grad()
        tloss(tmodel(Xtr_t[b]), ytr_t[b]).backward()
        topt.step()
with torch.no_grad():
    tacc = (tmodel(torch.tensor(X_te, dtype=torch.float32)).argmax(1).numpy() == y_te).mean()
print(f"PyTorch 测试准确率(5轮): {tacc*100:.2f}%   |   minitorch: {hist['acc'][-1]*100:.2f}%")

## 📦 沉淀进 minitorch

本节没有新增框架代码——我们是在**使用**前面造好的 `Tensor / nn`。但你可能注意到训练循环里仍有样板：手动遍历参数做 `p.data -= lr*p.grad`、手动切 mini-batch。

下一个 Part 我们就把这些封装掉。

## 小练习

1. **加宽/加深**：把模型改成 `784→256→128→10`（两个隐藏层），准确率能提升吗？
2. **学习率**：试 `lr=0.01` 和 `lr=1.0`，观察收敛速度与稳定性。
3. **看错样本**：找出测试集中被分错的样本并可视化，它们是不是确实"长得像"别的数字？

## 小结 & 下一站

✅ 我们理解了初始化的重要性，并用 minitorch **从零训练出一个 95%+ 准确率的 MNIST 分类器**——这是框架的第一个完整实战！

**下一站 → Part 5 `11_optimizers_sgd_momentum_adam`**：把"参数更新"封装成优化器（`SGD / Momentum / RMSProp / Adam`），再配上 `DataLoader`、`Dropout`、`BatchNorm`——让训练既优雅又强大。